# einops-repeat-broadcast — faded example 1: Broadcast a token-type embedding across the batch

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `einops-repeat-broadcast`. Running the beacon reports progress on the `Einops: Repeat-as-broadcast` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Repeat-as-broadcast` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-repeat-broadcast`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-repeat-broadcast"
DD_SUBTOPIC = "Einops: Repeat-as-broadcast"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

`einops.repeat` inserts new broadcast axes with stride zero, so a `(T, D)` per-position table can be viewed as `(B, T, D)` to add into a batched hidden state without copying. The new leading `b` axis is the broadcast — every batch element re-reads the same `(T, D)` table.

## Faded exercise 1

Implement `broadcast_type_emb(type_emb, hidden)`. `type_emb` has shape `(T, D)` (one embedding per sequence position) and `hidden` has shape `(B, T, D)` (the batched hidden state). Return `type_emb` broadcast to `(B, T, D)` with `einops.repeat`, inserting the new leading batch axis as a stride-zero view. Complete the missing `repeat` call.

**Fill in:** the einops.repeat call that inserts the leading batch axis to make a (B, T, D) view

In [ ]:
import torch as t
import einops
from einops import repeat

t.manual_seed(3)
type_emb = t.randn(5, 8)
hidden = t.randn(4, 5, 8)

def broadcast_type_emb(type_emb, hidden):
    B = hidden.shape[0]
    emb_b = repeat(type_emb, 't d -> b t d', b=B)
    return emb_b

emb_b = broadcast_type_emb(type_emb, hidden)
print(emb_b.shape)


def _test():
    emb_b = broadcast_type_emb(type_emb, hidden)
    assert emb_b.shape == (4, 5, 8), emb_b.shape
    # every batch slice must equal the original (T, D) table
    for b in range(4):
        assert t.equal(emb_b[b], type_emb), f'batch {b} differs'
    # stride-zero on the broadcast axis -> shares storage, no copy
    assert emb_b.data_ptr() == type_emb.data_ptr()
    # broadcast view must add cleanly into the hidden state
    summed = hidden + emb_b
    assert summed.shape == hidden.shape


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import einops
from einops import repeat

t.manual_seed(3)
type_emb = t.randn(5, 8)
hidden = t.randn(4, 5, 8)

def broadcast_type_emb(type_emb, hidden):
    B = hidden.shape[0]
    emb_b = repeat(type_emb, 't d -> b t d', b=B)
    return emb_b

emb_b = broadcast_type_emb(type_emb, hidden)
print(emb_b.shape)
```
</details>